
Mount G-drive

In [1]:
# ── Install ────────────────────────────────────────────────
!pip install transformers torch h5py tqdm -q

# ── Mount Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:


import os, torch, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

# ── Config ─────────────────────────────────────────────────
INPUT_FILE  = "/content/drive/MyDrive/wiki-samples/wiki_0_sentences.txt"   # ← your source text file
OUTPUT_DIR  = "/content/drive/MyDrive/wiki_embeddings"

# Replace with:
# INPUT_FILE = "./data/wiki_0_sentences.txt"
# for multiple files
# INPUT_FILES = [
#     "./data/wiki_0_sentences.txt",
#     "./data/wiki_1_sentences.txt",
#     "./data/wiki_2_sentences.txt",
# ]  # ← path to their local .txt file
# OUTPUT_DIR  = "./wiki_embeddings" <- path to save embeddings
MAX_SEQ_LEN = 512
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = [
    ("bert_tiny",  "prajjwal1/bert-tiny",  128, 512),
    ("bert_mini",  "prajjwal1/bert-mini",  256, 512),
    ("bert_small", "prajjwal1/bert-small", 512, 256),
    ("bert_base",  "bert-base-uncased",    768, 128),
    # (save_name, hf_model_id, embed_dim, batch_size)
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────
def read_all_sentences(filepath):
    """Read and return all non-empty lines from the text file."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return [line.strip() for line in f if line.strip()]

@torch.no_grad()
def encode_cls(model, tokenizer, sentences):
    """CLS-token pooling: returns (batch_size, hidden_dim) float32 array."""
    enc = tokenizer(
        sentences, padding=True, truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = model(**enc)
    return out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)

# ── Load sentences once ────────────────────────────────────
print(f"Reading sentences from:\n  {INPUT_FILE}\n")
sentences = read_all_sentences(INPUT_FILE)
print(f"Total sentences loaded: {len(sentences):,}\n")

# Optionally save the full sentence list once (shared across models)
sentences_path = os.path.join(OUTPUT_DIR, "sentences_all.txt")
with open(sentences_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sentences))
print(f"Sentences saved → {sentences_path}\n")

# ── Generate embeddings for each model ────────────────────
for model_name, hf_id, embed_dim, batch_size in MODELS:
    print(f"{'='*60}")
    print(f"Model  : {model_name}  ({hf_id})")
    print(f"Dim    : {embed_dim}  |  Batch size: {batch_size}  |  Device: {DEVICE}")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model     = AutoModel.from_pretrained(hf_id).to(DEVICE).eval()

    all_embeddings = []
    batches = range(0, len(sentences), batch_size)

    for i in tqdm(batches, desc=f"  Encoding [{model_name}]"):
        batch = sentences[i : i + batch_size]
        embs  = encode_cls(model, tokenizer, batch)
        all_embeddings.append(embs)

    # Concatenate all batch outputs → (N, embed_dim)
    merged = np.concatenate(all_embeddings, axis=0)
    tensor = torch.from_numpy(merged).half()   # store as fp16 to halve disk usage

    save_path = os.path.join(OUTPUT_DIR, f"embeddings_{model_name}.pt")
    torch.save(tensor, save_path)

    file_mb = os.path.getsize(save_path) / 1e6
    print(f"  ✓ Saved  shape={tuple(tensor.shape)}  →  {save_path}  ({file_mb:.1f} MB)\n")

    # Free GPU/CPU memory before loading the next model
    del model, tokenizer, all_embeddings, merged, tensor
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("✅  All models done.")

Reading sentences from:
  /content/drive/MyDrive/wiki-samples/wiki_0_sentences.txt

Total sentences loaded: 1,999,993

Sentences saved → /content/drive/MyDrive/wiki_embeddings/sentences_all.txt

Model  : bert_tiny  (prajjwal1/bert-tiny)
Dim    : 128  |  Batch size: 512  |  Device: cuda


pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

  Encoding [bert_tiny]:   0%|          | 0/3907 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]


  Encoding [bert_tiny]:  35%|███▍      | 1366/3907 [03:50<07:09,  5.92it/s]


KeyboardInterrupt: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA



# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────


COMPRESS_DIMS = [128, 256, 512]     # BERT Base 768D → target dims

SMALL_MODELS = [                    # native small-BERT embeddings to compare
    ("bert_tiny",  128),
    ("bert_mini",  256),
    ("bert_small", 512),
]

CONFIG = {
    'input_dim':       768,
    'epochs':          40,
    'batch_size':      2048,
    'lr':              1e-3,
    'tau':             0.05,
    'dropout_rate':    0.1,
    'moco_queue_size': 65536,
}


# ─────────────────────────────────────────────────────────────────────────────
# 1. ADAPTIVE ENCODER
# ─────────────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, input_dim=768, latent_dim=128, dropout_rate=0.1):
        super().__init__()
        dims = [input_dim]
        d = input_dim
        while d > latent_dim * 2:
            d = max(d // 2, latent_dim * 2)
            dims.append(d)

        # Ensure at least one hidden layer so dropout augmentation actually happens
        if len(dims) == 1 or (len(dims) == 2 and dims[1] == latent_dim):
            dims.insert(-1, max(input_dim // 2, latent_dim + 1))

        dims.append(latent_dim)

        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:      # no activation on final projection
                layers += [nn.LayerNorm(dims[i + 1]), nn.GELU(), nn.Dropout(dropout_rate)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# 2. MoCo QUEUE
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def momentum_update(model_q, model_k, m=0.999):
    for pq, pk in zip(model_q.parameters(), model_k.parameters()):
        pk.data = pk.data * m + pq.data * (1.0 - m)


class MoCoQueue:
    def __init__(self, size, dim, device):
        self.size   = size
        self.ptr    = 0
        # Initialize with zeros or random; it will be filled during first epoch
        self.buffer = F.normalize(torch.randn(size, dim, device=device), dim=1)

    def get(self):
        return self.buffer.detach()

    @torch.no_grad()
    def enqueue(self, z_norm):
        B = z_norm.size(0)
        end = self.ptr + B
        if end <= self.size:
            self.buffer[self.ptr:end] = z_norm
        else:
            tail = self.size - self.ptr
            self.buffer[self.ptr:]        = z_norm[:tail]
            self.buffer[:end - self.size] = z_norm[tail:]
        self.ptr = end % self.size


# ─────────────────────────────────────────────────────────────────────────────
# 3. kNN HELPERS (Optimized for Large N)
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_knn_indices(embeddings, k=10, batch_size=4096): # Increased batch size
    N = embeddings.size(0)
    emb = F.normalize(embeddings.to(device), p=2, dim=1)
    all_indices = []

    for i in range(0, N, batch_size):
        end = min(i + batch_size, N)
        sims = torch.matmul(emb[i:end], emb.T)

        for j in range(end - i):
            sims[j, i + j] = -1.0

        all_indices.append(torch.topk(sims, k=k, dim=1).indices.cpu())

        # Aggressive memory cleanup
        del sims
        torch.cuda.empty_cache()

    return torch.cat(all_indices, dim=0)

def knn_recall(gt_indices, pred_indices, k=10):
    """
    Memory-safe Recall@K calculation for large datasets.
    """
    gt_indices = gt_indices.numpy()
    pred_indices = pred_indices.numpy()

    total = 0.0
    for i in range(len(gt_indices)):
        intersection = np.intersect1d(gt_indices[i], pred_indices[i], assume_unique=True)
        total += len(intersection) / k

    return total / len(gt_indices)

# ─────────────────────────────────────────────────────────────────────────────
# 4. InfoNCE LOSS
# ─────────────────────────────────────────────────────────────────────────────
def simcse_info_nce(z_q, z_k, queue, tau):
    z_q_n = F.normalize(z_q, p=2, dim=1)
    z_k_n = F.normalize(z_k, p=2, dim=1)

    # Positive pair: (batch, 1)
    pos = torch.sum(z_q_n * z_k_n, dim=1, keepdim=True) / tau
    # Negative pairs: (batch, queue_size)
    neg = torch.matmul(z_q_n, queue.T) / tau

    logits = torch.cat([pos, neg], dim=1)
    labels = torch.zeros(z_q.size(0), dtype=torch.long, device=z_q.device)
    return F.cross_entropy(logits, labels)


# ─────────────────────────────────────────────────────────────────────────────
# 5. ENCODE HELPER
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def encode_latent(model, embeddings, batch_size):
    model.eval()
    parts = []
    for i in range(0, len(embeddings), batch_size):
        batch = embeddings[i:i + batch_size].to(device)
        parts.append(model(batch).cpu())
    return torch.cat(parts, dim=0)


# ─────────────────────────────────────────────────────────────────────────────
# 6. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train_simcse_moco(X, latent_dim, config):
    print(f"    Training SimCSE+MoCo  768D → {latent_dim}D")

    model_q = Encoder(
        input_dim    = config['input_dim'],
        latent_dim   = latent_dim,
        dropout_rate = config['dropout_rate'],
    ).to(device)
    if torch.cuda.device_count() > 1:
        model_q = nn.DataParallel(model_q)

    loader = DataLoader(
        TensorDataset(X),
        batch_size  = config['batch_size'],
        shuffle     = True,
        num_workers = 8,  # Increased from 2 to match cluster CPU cores
        pin_memory  = True,
    )

    optimizer = optim.Adam(model_q.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    # Momentum Encoder
    model_k = copy.deepcopy(model_q).to(device)
    for p in model_k.parameters():
        p.requires_grad = False

    queue = MoCoQueue(size=config['moco_queue_size'], dim=latent_dim, device=device)

    for epoch in range(config['epochs']):
        model_q.train()
        epoch_loss = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)

            # Query view
            z_q = model_q(X_batch)

            with torch.no_grad():
                momentum_update(model_q, model_k, m=0.999)
                model_k.train() # Keep dropout active for SimCSE augmentation
                z_k = model_k(X_batch)

            loss = simcse_info_nce(z_q, z_k, queue.get(), config['tau'])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Enqueue the key view
            queue.enqueue(F.normalize(z_k.detach(), p=2, dim=1))
            epoch_loss += loss.item()

        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"      Epoch {epoch+1:3d}/{config['epochs']} | Loss: {epoch_loss/len(loader):.4f}")

    return model_q


# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────

# ── Load BERT Base 768D ───────────────────────────────────────────────────────
print("Loading BERT Base 768D embeddings ...")
bert_base = torch.load(f'{OUTPUT_DIR}/embeddings_bert_base.pt', weights_only=True).float()
N = len(bert_base)
print(f"  Shape: {bert_base.shape}")

# ── Ground truth kNN from full 768D ──────────────────────────────────────────
print("\nComputing GT kNN (BERT Base 768D) ...")
knn_gt = compute_knn_indices(bert_base, k=10)
print("  Done.")

results = []

# ─────────────────────────────────────────────────────────────────────────────
# PART A  —  Small BERT models
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART A : Small BERT models (raw native embeddings)")
print("=" * 60)

for model_name, dim in SMALL_MODELS:
    print(f"\n  [{model_name}]  {dim}D ...")
    emb = torch.load(f'{OUTPUT_DIR}/embeddings_{model_name}.pt', weights_only=True).float()[:N]
    knn    = compute_knn_indices(emb, k=10)
    recall = knn_recall(knn_gt, knn, k=10)
    print(f"    Recall@10 = {recall:.4f}")
    results.append({'method': model_name, 'dim': dim, 'recall': recall})

# ─────────────────────────────────────────────────────────────────────────────
# PART B  —  PCA compression
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B : PCA compression of BERT Base 768D")
print("=" * 60)

for latent_dim in COMPRESS_DIMS:
    print(f"\n  PCA  768D → {latent_dim}D ...")
    pca = PCA(n_components=latent_dim)
    # Use float64 for PCA stability, then back to float32 for torch
    pca_emb = torch.tensor(
        pca.fit_transform(bert_base.numpy()),
        dtype=torch.float32
    )
    print(f"    Explained variance : {pca.explained_variance_ratio_.sum():.4f}")
    knn    = compute_knn_indices(pca_emb, k=10)
    recall = knn_recall(knn_gt, knn, k=10)
    print(f"    Recall@10 = {recall:.4f}")
    results.append({'method': f'PCA 768→{latent_dim}', 'dim': latent_dim, 'recall': recall})

# ─────────────────────────────────────────────────────────────────────────────
# PART C  —  Contrastive compression (SimCSE + MoCo)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART C : Contrastive compression of BERT Base 768D")
print("=" * 60)

for latent_dim in COMPRESS_DIMS:
    print(f"\n  Contrastive 768D → {latent_dim}D ...")

    # 1. Train the encoder
    model = train_simcse_moco(bert_base, latent_dim, CONFIG)

    # 2. Generate compressed embeddings for the whole set
    compressed_emb = encode_latent(model, bert_base, batch_size=CONFIG['batch_size'])

    # 3. Evaluate via kNN Recall
    knn    = compute_knn_indices(compressed_emb, k=10)
    recall = knn_recall(knn_gt, knn, k=10)
    print(f"    Recall@10 = {recall:.4f}")

    results.append({'method': f'Contrastive 768→{latent_dim}', 'dim': latent_dim, 'recall': recall})

    # Cleanup GPU memory between different dimension runs
    del model
    torch.cuda.empty_cache()

# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"{'Method':<25} | {'Dim':<10} | {'Recall@10':<10}")
print("-" * 60)
for r in results:
    print(f"{r['method']:<25} | {r['dim']:<10} | {r['recall']:.4f}")
print("=" * 60)